# Notebook 07: Baseline Comparison : EfficientNet-B0 vs ConvNeXt-Nano vs Swin-T
This notebook performs a unified comparative analysis across all three evaluated backbones (**EfficientNet-B0**, **ConvNeXt-Nano**, and **Swin-T**). 

## 1. Setup & Environment
Imports core data processing and plotting libraries (`matplotlib`, `numpy`, `json`) and sets up working output directories.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

OUT = Path("/kaggle/working")
print("Output dir:", OUT)

## 2. Load Cross-Validation Artifacts
Validates that all three cross-validation result JSON files (`effnet_b0`, `convnext_nano`, and `swin_t`) exist before parsing.

In [ ]:
NB06 = Path("/kaggle/input/notebooks/mfjmrizvi/06-real-5-fold-cv")  

EFFNET_JSON   = NB06 / "effnet_b0_real_cv_results.json"
CONVNEXT_JSON = NB06 / "convnext_nano_real_cv_results.json"
SWINT_JSON    = NB06 / "swin_t_real_cv_results.json"

for p in [EFFNET_JSON, CONVNEXT_JSON, SWINT_JSON]:
    print(f"{'✓' if p.exists() else '✗ MISSING'} {p.name}")

### 2.1 Parse Metrics & Feature Dimensions
Loads each model's JSON file, attaches known architectural feature dimensions (`FEAT_DIMS`), and prints fold-level and aggregated ($\text{Mean} \pm \text{SD}$) metrics for initial inspection.

In [ ]:
with open(EFFNET_JSON) as f: effnet = json.load(f)
with open(CONVNEXT_JSON) as f: convnext = json.load(f)
with open(SWINT_JSON) as f: swint = json.load(f)

# NOTE: the new JSONs don't have a "feat_dim" key 
# adding it manually since it's architecture-fixed and known.
FEAT_DIMS = {"EfficientNet-B0": 1280, "ConvNeXt-Nano": 640, "Swin-T": 768}

results = {
    "EfficientNet-B0" : effnet,
    "ConvNeXt-Nano"   : convnext,
    "Swin-T"          : swint,
}

for name, r in results.items():
    r["feat_dim"] = FEAT_DIMS[name]

print("Loaded results for:", list(results.keys()))

In [ ]:
for name, r in results.items():
    print(f"\n{'='*50}\n  {name}\n{'='*50}")
    print(f"  FEAT_DIM : {r['feat_dim']}")

    print("\n  Fold results:")

    for fold in r["fold_results"]:
        print(f"\n    Fold {fold['fold']}:")
        print(f"      Threshold   : {fold['threshold']:.4f}")
        print(f"      AUC         : {fold['auc']:.4f}")
        print(f"      F1          : {fold['f1']:.4f}")
        print(f"      Sensitivity : {fold['sensitivity']:.4f}")
        print(f"      Specificity : {fold['specificity']:.4f}")
        print(f"      FPR         : {fold['fpr']:.4f}")
        print(f"      MCC         : {fold['mcc']:.4f}")
        print(f"      ECE         : {fold['ece']:.4f}")

    print("\n  CV Mean:")
    for k, v in r["cv_mean"].items():
        print(f"    {k:15s}: {v:.4f}")

    print("\n  CV Std:")
    for k, v in r["cv_std"].items():
        print(f"    {k:15s}: {v:.4f}")

## 3. Unified Cross-Validation Metrics Table

Generates a formatted summary table comparing key metrics ($\text{Mean} \pm \text{SD}$) across all 5 cross-validation folds for each backbone architecture.

In [ ]:
# Build CV comparison table with Mean ± SD
models = list(results.keys())
col_width = 22

metrics = [
    ("Threshold", "threshold"),
    ("AUC", "auc"),
    ("F1", "f1"),
    ("Sensitivity", "sensitivity"),
    ("Specificity", "specificity"),
    ("FPR", "fpr"),
    ("MCC", "mcc"),
    ("ECE", "ece"),
]

header = f"{'Metric':<20}" + "".join(
    f"{m:>{col_width}}" for m in models
)

print("\n" + "=" * (20 + col_width * len(models)))
print(header)
print("=" * (20 + col_width * len(models)))

# FEAT_DIM
print(f"{'FEAT_DIM':<20}" + "".join(
    f"{results[m]['feat_dim']:>{col_width}d}" for m in models
))

for label, key in metrics:
    row = f"{label:<20}"

    for name in models:

        # Threshold is stored per fold, not in cv_mean/cv_std
        if key == "threshold":
            values = [
                fold["threshold"]
                for fold in results[name]["fold_results"]
            ]

            mean = np.mean(values)
            std = np.std(values, ddof=1)

        # Other metrics already have CV mean and SD
        else:
            mean = results[name]["cv_mean"][key]
            std = results[name]["cv_std"][key]

        value = f"{mean:.4f} ± {std:.4f}"
        row += f"{value:>{col_width}}"

    print(row)

print("=" * (20 + col_width * len(models)))

## 4. Primary Performance Visualizations

### 4.1 Multi-Metric Performance Overview
Generates a side-by-side bar chart comparing overall **AUC**, **F1-Score**, **Sensitivity**, and **Specificity** across the three architectures.

In [ ]:
# Bar chart comparison for 5-fold CV metrics

cv_metrics_to_plot = ["auc", "f1", "sensitivity", "specificity"]
metric_labels = ["AUC", "F1", "Sensitivity", "Specificity"]

colors = ["#2196F3", "#FF9800", "#4CAF50"]

x = np.arange(len(cv_metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))

for i, (name, color) in enumerate(zip(models, colors)):

    vals = [
        results[name]["cv_mean"][m]
        for m in cv_metrics_to_plot
    ]

    bars = ax.bar(
        x + i * width,
        vals,
        width,
        label=name,
        color=color,
        alpha=0.85
    )

    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=7
        )

ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title(
    "5-Fold Cross-Validation Performance\n"
    "(Mean across folds)"
)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    OUT / "nb07_cv_bar_chart.png",
    dpi=150
)

plt.show()

print("Saved nb07_cv_bar_chart.png")

### 4.2 Cross-Validation Area Under the Curve (AUC)
Renders a targeted comparison of overall classification power via mean 5-fold CV AUC scores.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
cv_aucs = [
    results[n]["cv_mean"]["auc"]
    for n in models
]

bars = ax.bar(
    models,
    cv_aucs,
    color=colors,
    alpha=0.85
)

for bar, val in zip(bars, cv_aucs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_ylim(0, 1.08)
ax.set_ylabel("AUC")
ax.set_title("5-Fold Cross-Validation AUC Comparison")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    OUT / "nb07_cv_auc_comparison.png",
    dpi=150
)

plt.show()

print("Saved nb07_cv_auc_comparison.png")

### 4.3 Calibrated Decision Thresholds
Visualises the mean operating threshold chosen across folds to maintain $\ge 90\%$ Sensitivity compared against the default $0.50$ decision threshold.

In [ ]:
thresholds = [
    np.mean([
        fold["threshold"]
        for fold in results[n]["fold_results"]
    ])
    for n in models
]

fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(
    models,
    thresholds,
    color=colors,
    alpha=0.85
)

for bar, val in zip(bars, thresholds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.axhline(
    0.5,
    color="red",
    linestyle="--",
    lw=1.2,
    label="Default threshold (0.5)"
)

ax.set_ylim(0, 1.1)
ax.set_ylabel("Mean CV Threshold")
ax.set_title(
    "Mean Classification Threshold per Model\n"
    "(mean across 5 CV folds)"
)

ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    OUT / "nb07_threshold_comparison.png",
    dpi=150
)

plt.show()

print("Saved nb07_threshold_comparison.png")

In [ ]:
for name in models:
    thresholds = [
        fold["threshold"]
        for fold in results[name]["fold_results"]
    ]

    print(f"{name}:")
    print("  Fold thresholds:", [f"{x:.4f}" for x in thresholds])
    print(f"  Mean ± SD      : {np.mean(thresholds):.4f} ± {np.std(thresholds, ddof=1):.4f}")

### 4.4 Probability Calibration (ECE)
Evaluates **Expected Calibration Error (ECE)** across models. Lower values indicate better-aligned predicted probabilities relative to true positive rates.

In [ ]:
cv_eces = [results[n]["cv_mean"]["ece"] for n in models]

fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(
    models,
    cv_eces,
    color=colors,
    alpha=0.85
)

for bar, v in zip(bars, cv_eces):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        v + 0.001,
        f"{v:.4f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

ax.set_xticks(range(len(models)))
ax.set_xticklabels(models)
ax.set_ylabel("Expected Calibration Error")
ax.set_title("Calibration Quality: 5-Fold CV ECE")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.savefig(
    OUT / "nb07_ece_comparison.png",
    dpi=150
)

plt.show()

## 5. Direct Comparative Summary

Prints a clean tabular comparison highlighting key metrics at the patient/bag level across all 5 cross-validation folds.

In [ ]:
print("\nThis Project Results\n")

comparison_results = [
    ("EfficientNet-B0", results["EfficientNet-B0"]),
    ("ConvNeXt-Nano",   results["ConvNeXt-Nano"]),
    ("Swin-T",          results["Swin-T"]),
]

col_w2 = [20, 14, 14, 16, 16, 14, 14]

headers2 = [
    "Model", "AUC", "F1",
    "Sensitivity", "Specificity", "FPR", "ECE"
]

print("".join(
    f"{h:<{w}}"
    for h, w in zip(headers2, col_w2)
))

print("-" * sum(col_w2))

for name, r in comparison_results:

    cm = r["cv_mean"]

    print(
        f"{name:<20}"
        f"{cm['auc']:<14.4f}"
        f"{cm['f1']:<14.4f}"
        f"{cm['sensitivity']:<16.4f}"
        f"{cm['specificity']:<16.4f}"
        f"{cm['fpr']:<14.4f}"
        f"{cm['ece']:<14.4f}"
    )

print("-" * sum(col_w2))

print("\nEval unit     : Patient/bag-level")
print("Eval strategy : 5-fold cross-validation")
print("Metrics       : Mean across 5 folds")

## 6. Export Baseline Summary

Exports the consolidated baseline metrics, threshold statistics and fold-level data into a single summary JSON file (`nb07_baseline_summary.json`) for downstream analysis.

In [ ]:
summary = {}

for name, r in results.items():

    # Extract fold thresholds because threshold is stored per fold
    fold_thresholds = [
        fold["threshold"]
        for fold in r["fold_results"]
    ]

    summary[name] = {
        "feat_dim": r["feat_dim"],

        # Threshold statistics across folds
        "threshold_mean": float(np.mean(fold_thresholds)),
        "threshold_std": float(np.std(fold_thresholds, ddof=1)),

        # 5-fold CV mean
        "cv_mean": {
            key: float(value)
            for key, value in r["cv_mean"].items()
        },

        # 5-fold CV standard deviation
        "cv_std": {
            key: float(value)
            for key, value in r["cv_std"].items()
        },

        # Individual fold results
        "fold_results": r["fold_results"],
    }

with open(OUT / "nb07_baseline_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved: nb07_baseline_summary.json")

## 7. Final Model Rankings & Selection

Ranks the three backbones across four primary criteria: **AUC** (discriminative power), **F1** (balance), **ECE** (calibration quality) and **FPR** (false positive rate at fixed sensitivity).

In [ ]:
# Final ranking printout (5-fold CV results)

print("\n" + "=" * 60)
print("  NB07: LEAK-FREE BASELINE COMPARISON SUMMARY")
print("=" * 60)

# ---------------------------------------------------------
# Ranking by CV AUC (higher is better)
# ---------------------------------------------------------
print("\n  Ranking by CV AUC:")

ranked = sorted(
    models,
    key=lambda n: results[n]["cv_mean"]["auc"],
    reverse=True
)

for i, name in enumerate(ranked, 1):
    auc = results[name]["cv_mean"]["auc"]
    print(f"    {i}. {name:<20} CV AUC = {auc:.4f}")


# ---------------------------------------------------------
# Ranking by CV F1 (higher is better)
# ---------------------------------------------------------
print("\n  Ranking by CV F1:")

ranked_f1 = sorted(
    models,
    key=lambda n: results[n]["cv_mean"]["f1"],
    reverse=True
)

for i, name in enumerate(ranked_f1, 1):
    f1 = results[name]["cv_mean"]["f1"]
    print(f"    {i}. {name:<20} CV F1  = {f1:.4f}")


# ---------------------------------------------------------
# Ranking by CV ECE (lower is better)
# ---------------------------------------------------------
print("\n  Ranking by CV ECE (lower is better):")

ranked_ece = sorted(
    models,
    key=lambda n: results[n]["cv_mean"]["ece"]
)

for i, name in enumerate(ranked_ece, 1):
    ece = results[name]["cv_mean"]["ece"]
    print(f"    {i}. {name:<20} CV ECE = {ece:.4f}")


# ---------------------------------------------------------
# Ranking by CV FPR (lower is better)
# ---------------------------------------------------------
print("\n  Ranking by CV FPR (lower is better):")

ranked_fpr = sorted(
    models,
    key=lambda n: results[n]["cv_mean"]["fpr"]
)

for i, name in enumerate(ranked_fpr, 1):
    fpr = results[name]["cv_mean"]["fpr"]
    print(f"    {i}. {name:<20} CV FPR = {fpr:.4f}")


print("=" * 60)